# 🟡 Interview: Implement Contrastive Loss

InfoNCE loss used in CLIP and SimCLR

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def contrastive_loss(q, k, temperature=0.07):
    """
    手写 InfoNCE 对比损失 —— 面试版。

    面试考点:
    1. InfoNCE = 目标为对角线的交叉熵
    2. 温度 τ 的作用: 越小分布越尖，越大越平滑
    3. log-sum-exp 技巧: logsumexp(x) = max(x) + log(Σ exp(x - max(x)))
    4. 对角线索引: logits[arange(N), arange(N)]

    参数: q: (N, D), k: (N, D), temperature: float
    返回: scalar
    """
    N = q.shape[0]

    # ---- Step 1: 相似度矩阵 ----
    logits = (q @ k.T) / temperature  # (N, N)
    # logits[i,j] = q[i]·k[j] / τ
    # 对角线 = 正样本对, 非对角线 = 负样本对

    # ---- Step 2: 数值稳定的 log-sum-exp ----
    # logsumexp(x) = max(x) + log(Σ exp(x - max(x)))
    # 先减最大值防溢出，再 exp 求和，最后加回最大值
    max_logits = logits.max(dim=-1, keepdim=True).values  # (N, 1)
    log_sum_exp = max_logits.squeeze(-1) + torch.log(
        torch.exp(logits - max_logits).sum(dim=-1)
    )  # (N,)

    # ---- Step 3: 正样本的 logit ----
    pos_logits = logits[torch.arange(N), torch.arange(N)]  # (N,)
    # 每个 q[i] 与其匹配 k[i] 的相似度

    # ---- Step 4: 交叉熵损失 ----
    # log_p[i] = pos_logits[i] - log_sum_exp[i]
    #          = log(exp(pos) / Σ exp(all))
    log_p = pos_logits - log_sum_exp  # (N,)

    return -log_p.mean()

In [ ]:
torch.manual_seed(0)
N, D = 8, 16

# Perfect alignment: q == k (normalized)
v = torch.randn(N, D)
q = v / v.norm(dim=-1, keepdim=True)
k = q.clone()
loss_perfect = contrastive_loss(q, k)
print(f"Perfect alignment loss: {loss_perfect:.4f}  (should be near log(1/N) = {-torch.log(torch.tensor(N, dtype=torch.float)):.4f})")

# Random embeddings should give higher loss
q_rand = torch.randn(N, D)
q_rand = q_rand / q_rand.norm(dim=-1, keepdim=True)
k_rand = torch.randn(N, D)
k_rand = k_rand / k_rand.norm(dim=-1, keepdim=True)
loss_rand = contrastive_loss(q_rand, k_rand)
print(f"Random embeddings loss:  {loss_rand:.4f}  (should be higher)")

In [ ]:
from torch_judge import check
check("contrastive_loss")

## 📝 核心思路总结

1. **InfoNCE = 对角线交叉熵**：`logits = q @ k.T / τ`，目标是 `[0,1,...,N-1]`，等价于让每个 q[i] 最匹配自己的 k[i]。
2. **温度 τ 的作用**：τ 小 → 分布尖锐 → 模型需要更高的置信度区分正负样本；τ 大 → 分布平滑 → 允许更多模糊。
3. **log-sum-exp 是数值稳定的关键**：`logsumexp(x) = max(x) + log(Σ exp(x - max(x)))`，直接 log(Σ exp(x)) 会溢出。
4. **批内负样本**：每个样本的负样本就是同 batch 的其他 N-1 个样本，batch 越大负样本越多，对比学习效果越好。